# Préprocessing des données SNCF

**Objectif :** Préparer les données de régularité des liaisons TGV pour leur analyse.

## 1. Chargement des librairies et des donnés

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
# Chargement des données brutes
df = pd.read_csv(
    "../data/raw/regularite-mensuelle-tgv-aqst.csv",
    sep=";"
)

print("Dimensions :", df.shape)
display(df.head())

# Création d'une copie de travail
df_clean = df.copy()

Dimensions : (12907, 26)


,Date,Service,Gare de départ,Gare d'arrivée,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Commentaire annulations,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,...,Nombre trains en retard > 15min,Retard moyen trains en retard > 15 (si liaison concurrencée par vol),Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141,870,5,NaN,289,11.247809,...,110,6.511118,44,8,36.134454,31.092437,10.924370,15.966387,5.042017,0.840336
1,2018-01,National,LE MANS,PARIS MONTPARNASSE,56,406,1,NaN,213,8.479969,...,32,5.363539,9,4,20.000000,35.000000,16.666667,16.666667,8.333333,3.333333
2,2018-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166,226,0,NaN,21,6.239683,...,11,2.938053,6,1,22.222222,27.777778,16.666667,16.666667,5.555556,11.111111
3,2018-01,National,PARIS MONTPARNASSE,NANTES,124,508,3,NaN,71,7.235211,...,39,5.292211,18,8,33.333333,22.222222,16.666667,20.370370,5.555556,1.851852
4,2018-01,National,POITIERS,PARIS MONTPARNASSE,94,472,4,NaN,224,6.784673,...,42,4.882372,10,0,15.789474,45.614035,19.298246,15.789474,1.754386,1.754386


## 2. Diagnostic et traitement des anomalies

## 2.1. Traitement des observations incohérentes

In [7]:
# Calcul du nombre de trains ayant effectivement circulé
df_clean["Trains circulés"] = (
    df_clean["Nombre de circulations prévues"]
    - df_clean["Nombre de trains annulés"]
)

display(df_clean["Trains circulés"].describe())

count    12907.000000
mean       265.169985
std        181.363943
min        -21.000000
25%        145.000000
50%        224.000000
75%        348.000000
max       1099.000000
Name: Trains circulés, dtype: float64

In [6]:
# Identification des observations avec des effectifs incohérents
anomalies = (
    (df_clean["Trains circulés"] <= 0)
    | (df_clean["Nombre de trains en retard à l'arrivée"] < 0)
    | (
        df_clean["Nombre de trains en retard à l'arrivée"]
        > df_clean["Trains circulés"]
    )
)

print("Observations concernées :", anomalies.sum())
print("Part des observations (%) :", round(anomalies.mean() * 100, 2))

Observations concernées : 73
Part des observations (%) : 0.57


In [8]:
# Exclusion des observations présentant des effectifs incohérents
df_clean = df_clean.loc[~anomalies].copy()

print("Observations initiales :", len(df))
print("Observations conservées :", len(df_clean))
print("Observations exclues :", len(df) - len(df_clean))

Observations initiales : 12907
Observations conservées : 12834
Observations exclues : 73


In [9]:
# Vérification de la cohérence des effectifs après nettoyage
print("Circulations non positives :", (df_clean["Trains circulés"] <= 0).sum())

print("Retards à l'arrivée négatifs :",
      (df_clean["Nombre de trains en retard à l'arrivée"] < 0).sum())

print("Retards supérieurs aux circulations :",
      (df_clean["Nombre de trains en retard à l'arrivée"] > df_clean["Trains circulés"]).sum())

Circulations non positives : 0
Retards à l'arrivée négatifs : 0
Retards supérieurs aux circulations : 0


Le contrôle des effectifs a permis d'identifier 73 observations non exploitables, représentant 0,57 % des données initiales. Ces observations ont été exclues de la base de travail afin de garantir la cohérence du calcul des taux de retard à l'arrivée. Le fichier source a été conservé sans modification.

## 2.2. Traitement des valeurs négatives dans les retards

In [10]:
# Identification des valeurs négatives dans les seuils de retard
colonnes_retard = [
    "Nombre trains en retard > 15min",
    "Nombre trains en retard > 30min",
    "Nombre trains en retard > 60min"
]

print((df_clean[colonnes_retard] < 0).sum())

Nombre trains en retard > 15min     0
Nombre trains en retard > 30min    43
Nombre trains en retard > 60min     0
dtype: int64


In [11]:
# Examen des observations présentant des retards négatifs
retards_negatifs = (df_clean[colonnes_retard] < 0).any(axis=1)

print("Observations concernées :", retards_negatifs.sum())

display(
    df_clean.loc[
        retards_negatifs,
        ["Date", "Gare de départ", "Gare d'arrivée"] + colonnes_retard
    ]
)

Observations concernées : 43


,Date,Gare de départ,Gare d'arrivée,Nombre trains en retard > 15min,Nombre trains en retard > 30min,Nombre trains en retard > 60min
10326,2025-01,LAUSANNE,PARIS LYON,15,-4,9
10329,2025-01,DIJON VILLE,PARIS LYON,47,-1,30
10330,2025-01,DOUAI,PARIS NORD,21,-1,9
10357,2025-01,LE MANS,PARIS MONTPARNASSE,51,-1,30
10358,2025-01,MARSEILLE ST CHARLES,MARNE LA VALLEE,43,-2,17
10379,2025-01,LILLE,PARIS NORD,49,-1,18
10380,2025-01,LYON PART DIEU,LILLE,64,-44,21
10381,2025-01,LYON PART DIEU,MARNE LA VALLEE,42,-2,18
10382,2025-01,LYON PART DIEU,PARIS LYON,43,-3,17
10392,2025-01,QUIMPER,PARIS MONTPARNASSE,31,-1,14


Après le premier nettoyage, **43 observations présentent encore des valeurs négatives** dans la variable « Nombre trains en retard > 30min ». Ces anomalies sont concentrées entre janvier et mars 2025, avec des valeurs allant jusqu'à −44. En revanche, aucune valeur négative n'a été détectée pour les retards de plus de 15 et 60 minutes. Ces incohérences semblent donc spécifiques à la variable des retards de plus de 30 minutes et nécessitent un traitement particulier avant son utilisation dans les analyses.

In [12]:
# Traitement des valeurs négatives du nombre de retards supérieurs à 30 minutes
colonne = "Nombre trains en retard > 30min"

df_clean.loc[df_clean[colonne] < 0, colonne] = float("nan")

In [13]:
# Contrôle des valeurs négatives et manquantes après traitement
print("Valeurs négatives :", (df_clean[colonne] < 0).sum())
print("Valeurs manquantes :", df_clean[colonne].isna().sum())
print("Observations conservées :", len(df_clean))

Valeurs négatives : 0
Valeurs manquantes : 43
Observations conservées : 12834


Nous avons choisi de conserver les 43 observations présentant des valeurs négatives dans la variable « Nombre trains en retard > 30min », car les autres informations de ces lignes restent potentiellement exploitables. Supprimer ces observations entraînerait une perte inutile de données. Nous avons donc décidé de remplacer uniquement les valeurs incohérentes par `NaN`, afin de préserver les informations utiles tout en évitant de fausser les analyses.